> 在线讲义：[第 12 章 · 线代二：SVD、低秩近似与 LoRA、最小二乘、条件数](https://wikieden.github.io/math-and-eval-stats/lessons/12-svd-least-squares.html) · 本书《数学与实验统计》

# 第 12 章 · 线代二：SVD、低秩近似与 LoRA、最小二乘、条件数

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/12-svd-least-squares.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 SVD | UΣVᵀ | `svd_parts(A)` | σ = 5, 3 |
| 第二步 低秩 | 截断 SVD | `rank_k(A, 1)` | 误差 3.0 |
| 第二步 LoRA | r(d + k) | `lora_params(4096, 4096, 8)` | 65,536 / 16,777,216 |
| 第三步 最小二乘 | 正规方程 | `normal_equations(design(XS), YS)` | 1.96、1.10 |
| 第四步 条件数 | σ_max/σ_min | `np.linalg.cond(ILL)` | 40,002 |

例子：一个 3×2 矩阵做 SVD；4096×4096 的层加 LoRA；5 个点拟合直线；一个几乎共线的 2×2 方程组。
里程碑 ME-M8 的"权重矩阵 秩 → 误差"曲线留给自己画。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
A = np.array([[3.0, 2.0], [2.0, 3.0], [2.0, -2.0]])
XS = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
YS = np.array([1.1, 2.9, 5.2, 7.1, 8.8])  # 编的：大约 y = 2x + 1
ILL = np.array([[1.0, 1.0], [1.0, 1.0001]])

**函数 `svd_parts`**：m = U Σ Vᵀ，返回 (U, 奇异值, Vᵀ)，奇异值从大到小。

In [ ]:
def svd_parts(m: np.ndarray):
    """m = U Σ Vᵀ，返回 (U, 奇异值, Vᵀ)，奇异值从大到小。"""
    return np.linalg.svd(m, full_matrices=False)

**函数 `rank_k`**：截断 SVD：只留前 k 个奇异值，是 Frobenius 范数下最好的秩 k 近似（Eckart–Young）。

In [ ]:
def rank_k(m: np.ndarray, k: int) -> np.ndarray:
    """截断 SVD：只留前 k 个奇异值，是 Frobenius 范数下最好的秩 k 近似（Eckart–Young）。"""
    u, s, vt = svd_parts(m)
    return (u[:, :k] * s[:k]) @ vt[:k]

**函数 `lora_params`**：LoRA：ΔW = B A，B 是 d×r、A 是 r×k。返回 (LoRA 参数量, 原矩阵参数量)。

In [ ]:
def lora_params(d: int, k: int, r: int) -> tuple[int, int]:
    """LoRA：ΔW = B A，B 是 d×r、A 是 r×k。返回 (LoRA 参数量, 原矩阵参数量)。"""
    return r * (d + k), d * k

**函数 `design`**：直线拟合的设计矩阵 [x, 1]。

In [ ]:
def design(xs: np.ndarray) -> np.ndarray:
    """直线拟合的设计矩阵 [x, 1]。"""
    return np.column_stack([xs, np.ones_like(xs)])

**函数 `normal_equations`**：最小二乘的正规方程：θ = (XᵀX)⁻¹ Xᵀ y。

In [ ]:
def normal_equations(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """最小二乘的正规方程：θ = (XᵀX)⁻¹ Xᵀ y。"""
    return np.linalg.solve(x.T @ x, x.T @ y)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 SVD：任何矩阵 = 旋转 · 拉伸 · 旋转")
u, s, vt = svd_parts(A)
print(f"  A 的奇异值 {np.round(s, 4).tolist()}；UΣVᵀ 还原误差 {np.abs(u @ np.diag(s) @ vt - A).max():.1e}")
print(f"  奇异值的平方 = AᵀA 的特征值：{np.round(s**2, 4).tolist()} vs {np.round(np.linalg.eigvalsh(A.T @ A)[::-1], 4).tolist()}")

In [ ]:
print("\n第二步 低秩近似：只留最大的奇异值")
a1 = rank_k(A, 1)
print(f"  秩 1 近似的误差 ‖A − A₁‖_F = {np.linalg.norm(A - a1):.4f} = 第二个奇异值 {s[1]:.4f}")
print(f"  保留的能量 σ₁²/Σσ² = {s[0] ** 2 / (s**2).sum():.3f}")
lp, full = lora_params(4096, 4096, 8)
print(f"  LoRA r = 8 加在 4096×4096 上：{lp:,} 个参数，原矩阵 {full:,} 个，占 {lp / full:.2%}")

In [ ]:
print("\n第三步 最小二乘：拟合 5 个点")
x = design(XS)
th = normal_equations(x, YS)
print(f"  正规方程：斜率 {th[0]:.4f}，截距 {th[1]:.4f}")
print(f"  伪逆：{np.round(np.linalg.pinv(x) @ YS, 4).tolist()}；lstsq：{np.round(np.linalg.lstsq(x, YS, rcond=None)[0], 4).tolist()}")
r = YS - x @ th
print(f"  残差和每一列正交：Xᵀr = {np.round(x.T @ r, 10).tolist()}")

In [ ]:
print("\n第四步 条件数：输入抖一点，答案抖多少")
print(f"  ILL 的条件数 {np.linalg.cond(ILL):,.0f}")
b1, b2 = np.array([2.0, 2.0001]), np.array([2.0, 2.0002])
print(f"  b = [2, 2.0001] → x = {np.round(np.linalg.solve(ILL, b1), 4).tolist()}；b 的第二个数只多 0.0001 → x = {np.round(np.linalg.solve(ILL, b2), 4).tolist()}")

## 3. 代码里才看得到的

1. **`full_matrices=False`**：3×2 的矩阵，完整 SVD 的 U 是 3×3，多出来的一列对应奇异值 0，用不上；瘦 SVD 只返回 3×2。
2. **`u[:, :k] * s[:k]`**：用广播把每列乘上对应的奇异值，比构造对角矩阵再相乘省事。
3. **正规方程会把条件数平方**：XᵀX 的条件数是 X 的平方，X 本身病态时正规方程误差很大；`lstsq` 和 `pinv` 直接在 X 上做 SVD，没有这个问题。

## 4. 自己改一改

1. 把 `ILL` 的 1.0001 改成 1.1，条件数变成多少？b 再加 0.0001，解还会跳那么远吗？
2. 给 `YS` 的最后一个点加 10（一个离群值），斜率和截距怎么变？最小二乘为什么怕离群值？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 任何矩阵转拉转，奇异值平方是 AᵀA 的特征值；
> 留前 k 个最省事，误差就是丢掉的那几个；
> LoRA 两根瘦矩阵，参数 r 乘 d 加 k；
> 最小二乘是垂直投影，条件数大就加阻尼。